# Convolutional Wasserstein Distances

Reimplementation of Solomon et al. 2015,
*Convolutional Wasserstein Distances: Efficient Optimal Transportation on
Geometric Domains*.

Core library: `src/convolutional_wasserstein/`. Demos and plotting: `scripts/`.
CLI: `uv run convw2 --help`.


In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import plotly.subplots as sp
import scipy.linalg as slin
import trimesh

from convolutional_wasserstein import (
    VoxelMesh,
    bary_channels_to_rgb,
    color_channel_distributions,
    cotangent_laplacian,
    gaussian_kernel,
    gaussian_on_mesh,
    grid_barycenter,
    load_binary_image,
    load_color_image,
    mesh_heat_operator,
    normalize_mesh,
    opposite_vertices,
    wasserstein_barycenter,
)
from convolutional_wasserstein.convolution import heat_convolve, naive_gaussian_convolution
from convolutional_wasserstein.paths import IMAGES_DIR, MESHES_DIR, portrait_color_path
from scripts.assets import bilinear_coefs, ensure_demo_images
from scripts.parallel import parallel_barycenters
from scripts.viz import (
    distribution_to_binary,
    distribution_to_point_cloud,
    mesh_scalar_field_grid,
    point_cloud,
    voxel_cubes,
    voxel_isosurface,
)

ensure_demo_images()


## 1. Separable convolution vs. naive O(n^4)

The Sinkhorn loop only ever calls `K v`. With a Gaussian kernel on a grid
this is separable, so 2-D becomes two 1-D convolutions — `O(n^d)` instead of
`O(n^{2d})`. Below we benchmark the two on the same input.

In [ ]:
import time

# 2-D Gaussian convolution: naive O(n^4) is BLAS-accelerated `np.tensordot(K, u)`
# along each axis, so for tiny n it's faster than separable convolve1d on a kernel
# that grows with n. Pick n large enough — and gamma small enough — to actually
# expose the asymptotic O(n^d) vs O(n^{2d}) gap.
ns = [32, 48, 64, 96, 128, 192, 256, 384]
gamma = 0.001  # kernel radius ~ 6*sigma = 3*n*sqrt(gamma) ~ n/10
repeats = 5
classical, separable = [], []
for n in ns:
    u = np.random.rand(n, n)
    kernel = gaussian_kernel(n, gamma)
    # warmup
    heat_convolve(u, kernel); naive_gaussian_convolution(u, gamma)

    t0 = time.perf_counter()
    for _ in range(repeats): heat_convolve(u, kernel)
    separable.append((time.perf_counter() - t0) / repeats)

    t0 = time.perf_counter()
    for _ in range(repeats): naive_gaussian_convolution(u, gamma)
    classical.append((time.perf_counter() - t0) / repeats)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ns, classical, "o-", label="naive  O(n^4)")
ax.loglog(ns, separable, "o-", label="separable  O(n^2)  (kernel ~ n)")
ax.set_xlabel("n  (image side)"); ax.set_ylabel("seconds / convolution")
ax.set_title("2-D Gaussian convolution (gamma=0.001)")
ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.show()


## 2. 2-D shape barycenters

Bilinear-weighted Wasserstein barycenters of four binary shapes.

In [ ]:
shape_dir = IMAGES_DIR / "shapes"
shapes = {
    name: load_binary_image(shape_dir / f"shape{i}filled.png").ravel()
    for name, i in [("circle", 1), ("dots", 2), ("star", 3), ("fivestar", 4)]
}
mus = [shapes["circle"], shapes["star"], shapes["fivestar"], shapes["dots"]]
n = int(round(mus[0].size ** 0.5))

grid_size = 5
fig, axes = plt.subplots(grid_size, grid_size, figsize=(11, 11))
for idx, coef in enumerate(bilinear_coefs(grid_size)):
    i, j = divmod(idx, grid_size)
    bary = grid_barycenter(mus, coef, n=n, gamma=0.0015, iterations=3, sharpen=True)
    axes[i, j].imshow((bary.reshape(n, n) > 1e-6), cmap="binary")
    axes[i, j].set_axis_off()
plt.subplots_adjust(left=0, bottom=0, right=1, top=1, wspace=0, hspace=0)
plt.show()


## 3. Portrait morph (color)

Same algorithm on each RGB channel — no sharpening, just an interpolation
between two color distributions.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

monge_rgb = load_color_image(portrait_color_path("monge"))
kant_rgb = load_color_image(portrait_color_path("kantorovich"))
n = monge_rgb.shape[0]
monge_ch = color_channel_distributions(monge_rgb)
kant_ch = color_channel_distributions(kant_rgb)

n_frames = 21
coefs = [np.array([t, 1 - t]) for t in np.linspace(0, 1, n_frames)]
gamma, iterations = 2e-4, 80
barys_rgb = [
    parallel_barycenters(
        [monge_ch[c], kant_ch[c]], coefs, n, gamma=gamma, iterations=iterations, sharpen=False, workers=None
    )
    for c in range(3)
]

frames = [
    bary_channels_to_rgb([barys_rgb[c][i] for c in range(3)], n)
    for i in range(n_frames)
]
peak = max(frame.max() for frame in frames)
frames = [frame / peak for frame in frames]

fig, ax = plt.subplots(figsize=(4, 4))
ax.axis("off")
im = ax.imshow(frames[0], vmin=0, vmax=1)

def _update(i):
    im.set_data(frames[i])
    return [im]

anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=120, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 4. Voxelized solid-mesh barycenters

We voxelize each `.off` mesh into a 3-D occupancy distribution using a
Monte-Carlo `mesh.contains` pass, then run the same convolutional Sinkhorn —
the only difference is the heat kernel is 3-D separable.

In [ ]:
names = ["duck", "torus", "mushroom", "sphere_102"]
n_voxels = 50

meshes = []
for name in names:
    m = VoxelMesh.from_file(MESHES_DIR / f"{name}.off", n=n_voxels)
    m.voxelize()
    meshes.append(m)

point_cloud(distribution_to_point_cloud(meshes[0].distribution, scale=3)).show()
voxel_cubes(meshes[0].binary).show()


In [ ]:
mus = [m.binary_distribution for m in meshes]
grid_size = 4
coefs = bilinear_coefs(grid_size)
barys = parallel_barycenters(
    mus, coefs, n_voxels, gamma=0.0015, iterations=4, sharpen=False, workers=None
)

corner_rgb = np.array(
    [
        [0.93, 0.27, 0.27],
        [0.30, 0.69, 0.31],
        [0.13, 0.45, 0.86],
        [0.98, 0.75, 0.18],
    ]
)

fig = sp.make_subplots(
    rows=grid_size,
    cols=grid_size,
    specs=[[{"type": "scene"}] * grid_size] * grid_size,
    horizontal_spacing=0.005,
    vertical_spacing=0.005,
)
subplot = None
for idx, (coef, bary) in enumerate(zip(coefs, barys, strict=True)):
    i, j = divmod(idx, grid_size)
    rgb = np.clip(coef @ corner_rgb, 0, 1) * 255
    color = f"rgb({int(rgb[0])},{int(rgb[1])},{int(rgb[2])})"
    subplot, _ = voxel_isosurface(distribution_to_binary(bary), smooth=30, color=color)
    fig.add_trace(subplot.data[0], row=i + 1, col=j + 1)
fig.update_scenes(subplot.layout.scene.to_plotly_json())
fig.update_layout(
    height=1200,
    width=1200,
    paper_bgcolor="white",
    margin=dict(l=0, r=0, t=40, b=0),
    title=dict(text=" × ".join(names), x=0.5, xanchor="center"),
)
fig.show()


## 5. Heat-kernel barycenters on a triangle mesh

When the support is a curved mesh rather than a Euclidean grid, the Gaussian
kernel is no longer separable. The paper substitutes a single backward-Euler
step of the heat equation:

$$
H_t v = (D_a + tL)^{-1} D_a v
$$

with $L$ the cotangent Laplacian and $D_a$ the mass-lumped vertex areas.
We prefactor with Cholesky once and reuse it across iterations.

In [ ]:
mesh = trimesh.load(MESHES_DIR / "sphere_1300.off", process=False)
normalize_mesh(mesh)
L, areas = cotangent_laplacian(mesh)
gamma_mesh = 0.001
T = np.diag(areas) + (gamma_mesh / 2.0) * L.toarray()
Lcho = slin.cholesky(T, lower=True)
apply_kernel = mesh_heat_operator(Lcho)

src_a, src_b = opposite_vertices(mesh)
sigma = 0.12
g1 = gaussian_on_mesh(mesh, source=src_a, sigma=sigma)
g2 = gaussian_on_mesh(mesh, source=src_b, sigma=sigma)

mesh_scalar_field_grid(
    mesh,
    [g1, g2],
    rows=1,
    cols=2,
    subtitles=["source A", "source B"],
    title="Geodesic Gaussians on antipodal vertices",
    height=420,
    width=900,
).show()


In [ ]:
frame_ts = [0.0, 0.15, 0.3, 0.45, 0.55, 0.7, 0.85, 1.0]
barys = [
    wasserstein_barycenter(
        [g1, g2], [1 - t, t], area=areas, apply_kernel=apply_kernel, iterations=30, sharpen=False
    )
    for t in frame_ts
]

mesh_scalar_field_grid(
    mesh,
    barys,
    rows=2,
    cols=4,
    subtitles=[f"t={t:g}" for t in frame_ts],
    title="Heat-kernel barycenter transport on the sphere",
    height=900,
    width=1600,
).show()